# Task 01 – Dataset Understanding & Preprocessing
## VisDrone2019 Aerial Imagery Dataset

This notebook covers:
1. Dataset structure overview
2. Annotation format explanation
3. Class distribution & statistics
4. Key challenges
5. Preprocessing & augmentation strategy
6. Sample visualizations

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

# Dataset root — update this path
DATA_ROOT = '../data'
TRAIN_SPLIT = 'VisDrone2019-DET-train'
VAL_SPLIT   = 'VisDrone2019-DET-val'

print('Setup complete.')

## 1. Dataset Structure

```
VisDrone2019-DET-train/
├── images/          # .jpg drone images (1360×765 or 2000×1500)
└── annotations/     # .txt per-image annotation files

Annotation format (one object per line):
  bbox_left, bbox_top, bbox_width, bbox_height, score, category, truncation, occlusion
```

**10 original classes:**
ignored(0), pedestrian(1), people(2), bicycle(3), car(4),
van(5), truck(6), tricycle(7), awning-tricycle(8), bus(9), motor(10)

**We retain:** pedestrian(1) + people(2) → `person`, car(4) + van(5) → `car`

In [ ]:
# Count images and annotations in each split
splits = [TRAIN_SPLIT, VAL_SPLIT]
print(f"{'Split':35s} {'Images':>8s} {'Annotations':>12s}")
print('-' * 58)

for split in splits:
    img_dir = Path(DATA_ROOT) / split / 'images'
    ann_dir = Path(DATA_ROOT) / split / 'annotations'
    n_imgs = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_anns = len(list(ann_dir.glob('*.txt'))) if ann_dir.exists() else 0
    print(f"{split:35s} {n_imgs:>8,d} {n_anns:>12,d}")

In [ ]:
# Analyze full class distribution (all 10 classes)
VISDRONE_CLASSES = {
    0: 'ignored', 1: 'pedestrian', 2: 'people', 3: 'bicycle',
    4: 'car', 5: 'van', 6: 'truck', 7: 'tricycle',
    8: 'awning-tricycle', 9: 'bus', 10: 'motor'
}

ann_dir = Path(DATA_ROOT) / TRAIN_SPLIT / 'annotations'
class_counts = Counter()
box_widths, box_heights, box_areas = [], [], []
images_with_zero_objects = 0

ann_files = sorted(ann_dir.glob('*.txt'))[:1000]  # sample 1000 files

for ann_file in tqdm(ann_files, desc='Parsing annotations'):
    objects_in_image = 0
    with open(ann_file) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 6: continue
            x, y, bw, bh = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            score = int(parts[4])
            cat   = int(parts[5])
            if score == 0: continue  # ignored
            class_counts[VISDRONE_CLASSES.get(cat, str(cat))] += 1
            if bw > 0 and bh > 0:
                box_widths.append(bw)
                box_heights.append(bh)
                box_areas.append(bw * bh)
                objects_in_image += 1
    if objects_in_image == 0:
        images_with_zero_objects += 1

print('\nClass distribution (sample 1000 images):')
for cls, cnt in class_counts.most_common():
    print(f'  {cls:20s}: {cnt:6,d}')

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('VisDrone Dataset Analysis (Train Split – 1k image sample)', fontsize=13, fontweight='bold')

# Class bar chart
sorted_classes = class_counts.most_common()
names  = [c[0] for c in sorted_classes]
counts = [c[1] for c in sorted_classes]
colors = ['#00C853' if n in ('pedestrian','people') else '#FF6D00' if n in ('car','van') else '#607D8B' for n in names]
axes[0].barh(names, counts, color=colors, edgecolor='none')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='#00C853', label='→ person (kept)'),
    Patch(facecolor='#FF6D00', label='→ car (kept)'),
    Patch(facecolor='#607D8B', label='ignored'),
])

# Box size scatter
sample = np.random.choice(len(box_widths), min(3000, len(box_widths)), replace=False)
axes[1].scatter([box_widths[i] for i in sample],
                [box_heights[i] for i in sample],
                alpha=0.15, s=8, color='#2979FF')
axes[1].set_title('Bounding Box Sizes (px)')
axes[1].set_xlabel('Width (px)')
axes[1].set_ylabel('Height (px)')
axes[1].axvline(32, color='red', linestyle='--', label='32px threshold')
axes[1].axhline(32, color='red', linestyle='--')
axes[1].legend()

# Box area histogram
areas_k = [a / 1000 for a in box_areas]
axes[2].hist(areas_k, bins=50, color='#AA00FF', edgecolor='none', alpha=0.85)
axes[2].set_title('Bounding Box Area Distribution')
axes[2].set_xlabel('Area (thousand px²)')
axes[2].set_ylabel('Frequency')
median_area = np.median(areas_k)
axes[2].axvline(median_area, color='orange', linestyle='--', label=f'Median: {median_area:.1f}k px²')
axes[2].legend()
axes[2].set_xlim(0, 50)  # Focus on small objects

plt.tight_layout()
plt.savefig('../outputs/images/01_dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKey stats:')
print(f'  Median box area  : {np.median(box_areas):,.0f} px²  ({np.median(box_areas)**0.5:.0f}x{np.median(box_areas)**0.5:.0f} equivalent)')
print(f'  % boxes < 32px wide: {sum(1 for w in box_widths if w<32)/len(box_widths)*100:.1f}%')
print(f'  Images with 0 valid objects: {images_with_zero_objects}')

In [ ]:
# Visualize sample images with GT annotations
from src.dataset import visualize_samples  # if running from root

img_dir   = Path(DATA_ROOT) / TRAIN_SPLIT / 'images'
label_dir = Path(DATA_ROOT) / TRAIN_SPLIT / 'labels'  # after conversion
ann_dir_raw = Path(DATA_ROOT) / TRAIN_SPLIT / 'annotations'

VISDRONE_TO_OURS = {1:0, 2:0, 4:1, 5:1}
CLASS_COLORS = {0: (0, 255, 80), 1: (0, 160, 255)}
CLASS_LABELS = {0: 'person', 1: 'car'}

img_files = sorted(img_dir.glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('VisDrone – Ground Truth Annotations (person=green, car=orange)', fontsize=13, fontweight='bold')

for idx, img_file in enumerate(img_files):
    img = cv2.imread(str(img_file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ann_file = ann_dir_raw / img_file.with_suffix('.txt').name
    ax = axes[idx // 3][idx % 3]
    ax.imshow(img)

    person_cnt, car_cnt = 0, 0
    if ann_file.exists():
        with open(ann_file) as f:
            for line in f:
                p = line.strip().split(',')
                if len(p) < 6: continue
                bx,by,bw,bh = int(p[0]),int(p[1]),int(p[2]),int(p[3])
                score, cat = int(p[4]), int(p[5])
                if score == 0 or cat not in VISDRONE_TO_OURS or bw<=0 or bh<=0: continue
                cls_id = VISDRONE_TO_OURS[cat]
                color = '#00C853' if cls_id == 0 else '#FF6D00'
                rect = patches.Rectangle((bx,by), bw, bh, linewidth=0.8, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                if cls_id == 0: person_cnt += 1
                else: car_cnt += 1

    ax.set_title(f'👤 {person_cnt}   🚗 {car_cnt}   |   {img_file.name}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/images/01_sample_annotations.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Key Challenges

| Challenge | Impact | Mitigation |
|-----------|--------|------------|
| **Tiny objects** | Most bboxes are <20px tall — hard for standard detectors | Higher resolution training (640→1280), mosaic augmentation |
| **Dense scenes** | 50–200 objects per image, heavy occlusion | NMS tuning, copy-paste augmentation |
| **Class imbalance** | Pedestrians 3× more than cars | Weighted loss (handled by YOLOv8 automatically via focal loss) |
| **Altitude variance** | Objects shrink/grow dramatically with altitude | Scale jitter augmentation (0.5 factor) |
| **Viewpoint** | Top-down oblique views differ from standard datasets | Rotation augmentation, fine-tuning (not training from scratch) |

## 3. Preprocessing & Augmentation Strategy

**Annotation conversion:** VisDrone → YOLO normalized format (handled in `src/dataset.py`)

**Online augmentations during training (via Ultralytics):**
- Mosaic (1.0): combines 4 images → exposes model to more objects per step
- MixUp (0.15): alpha blending of two images
- Copy-paste (0.1): copy small objects and paste into other images
- Scale jitter (±50%): simulates altitude changes
- Horizontal + vertical flip
- HSV color jitter: handles varying lighting conditions
- Rotation (±10°): drone tilt

**Image size:** 640px (fast training), optionally 1280px for better small-object detection

In [ ]:
# Step: Convert annotations (run once)
import subprocess
result = subprocess.run(
    ['python', 'src/dataset.py', '--convert', '--data_root', DATA_ROOT],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

### ✅ Task 01 Complete

**Summary:**
- VisDrone has 10 classes; we retain `person` (pedestrian + people) and `car` (car + van)
- ~70% of bounding boxes are smaller than 32×32 pixels — small object detection is the core challenge
- Annotations converted to YOLO format; augmentation strategy designed for aerial imagery
- Ready for training in Task 02